In [1]:
import numpy as np
from oneibl.onelight import ONE
import numpy.random as npr
import json
from collections import defaultdict
import wget
from zipfile import ZipFile
import os
npr.seed(65)
# Continue preprocessing of IBL dataset and create design matrix for GLM-HMM
from sklearn import preprocessing
from scipy.stats import bernoulli
from oneibl.onelight import ONE
one = ONE()

In [2]:
def get_animal_name(eid):
    # get session id:
    raw_session_id = eid.split('Subjects/')[1]
    # Get animal:
    animal = raw_session_id.split('/')[0]
    return animal


def get_raw_data(eid):
    print(eid)
    # get session id:
    raw_session_id = eid.split('Subjects/')[1]
    # Get animal:
    animal = raw_session_id.split('/')[0]
    # replace '/' with dash in session ID
    session_id = raw_session_id.replace('/', '-')
    # hack to work with ONE:
    current_dir = os.getcwd()
    os.chdir("../../data/ibl/")
    # Get choice data, stim data and rewarded/not rewarded:
    choice = one.load_dataset(eid, '_ibl_trials.choice')
    stim_left = one.load_dataset(eid, '_ibl_trials.contrastLeft')
    stim_right = one.load_dataset(eid, '_ibl_trials.contrastRight')
    rewarded = one.load_dataset(eid, '_ibl_trials.feedbackType')
    bias_probs = one.load_dataset(eid, '_ibl_trials.probabilityLeft')
    os.chdir(current_dir)
    return animal, session_id, stim_left, stim_right, rewarded, choice, \
           bias_probs


def create_stim_vector(stim_left, stim_right):
    # want stim_right - stim_left
    # Replace NaNs with 0:
    stim_left = np.nan_to_num(stim_left, nan=0)
    stim_right = np.nan_to_num(stim_right, nan=0)
    # now get 1D stim
    signed_contrast = stim_right - stim_left
    return signed_contrast


def create_previous_choice_vector(choice):
    ''' choice: choice vector of size T
        previous_choice : vector of size T with previous choice made by
        animal - output is in {0, 1}, where 0 corresponds to a previous left
        choice; 1 corresponds to right.
        If the previous choice was a violation, replace this with the choice
        on the previous trial that was not a violation.
        locs_mapping: array of size (~num_viols)x2, where the entry in
        column 1 is the location in the previous choice vector that was a
        remapping due to a violation and the
        entry in column 2 is the location in the previous choice vector that
        this location was remapped to
    '''
    previous_choice = np.hstack([np.array(choice[0]), choice])[:-1]
    locs_to_update = np.where(previous_choice == -1)[0]
    locs_with_choice = np.where(previous_choice != -1)[0]
    loc_first_choice = locs_with_choice[0]
    locs_mapping = np.zeros((len(locs_to_update) - loc_first_choice, 2),
                            dtype='int')

    for i, loc in enumerate(locs_to_update):
        if loc < loc_first_choice:
            # since no previous choice, bernoulli sample: (not output of
            # bernoulli rvs is in {1, 2})
            previous_choice[loc] = bernoulli.rvs(0.5, 1) - 1
        else:
            # find nearest loc that has a previous choice value that is not
            # -1, and that is earlier than current trial
            potential_matches = locs_with_choice[
                np.where(locs_with_choice < loc)]
            absolute_val_diffs = np.abs(loc - potential_matches)
            absolute_val_diffs_ind = absolute_val_diffs.argmin()
            nearest_loc = potential_matches[absolute_val_diffs_ind]
            locs_mapping[i - loc_first_choice, 0] = int(loc)
            locs_mapping[i - loc_first_choice, 1] = int(nearest_loc)
            previous_choice[loc] = previous_choice[nearest_loc]
    assert len(np.unique(
        previous_choice)) <= 2, "previous choice should be in {0, 1}; " + str(
        np.unique(previous_choice))
    return previous_choice, locs_mapping


def create_wsls_covariate(previous_choice, success, locs_mapping):
    '''
    inputs:
    success: vector of size T, entries are in {-1, 1} and 0 corresponds to
    failure, 1 corresponds to success
    previous_choice: vector of size T, entries are in {0, 1} and 0
    corresponds to left choice, 1 corresponds to right choice
    locs_mapping: location remapping dictionary due to violations
    output:
    wsls: vector of size T, entries are in {-1, 1}.  1 corresponds to
    previous choice = right and success OR previous choice = left and
    failure; -1 corresponds to
    previous choice = left and success OR previous choice = right and failure
    '''
    # remap previous choice vals to {-1, 1}
    remapped_previous_choice = 2 * previous_choice - 1
    previous_reward = np.hstack([np.array(success[0]), success])[:-1]
    # Now need to go through and update previous reward to correspond to
    # same trial as previous choice:
    for i, loc in enumerate(locs_mapping[:, 0]):
        nearest_loc = locs_mapping[i, 1]
        previous_reward[loc] = previous_reward[nearest_loc]
    wsls = previous_reward * remapped_previous_choice
    assert len(np.unique(wsls)) == 2, "wsls should be in {-1, 1}"
    return wsls


def remap_choice_vals(choice):
    # raw choice vector has CW = 1 (correct response for stim on left),
    # CCW = -1 (correct response for stim on right) and viol = 0.  Let's
    # remap so that CW = 0, CCw = 1, and viol = -1
    choice_mapping = {1: 0, -1: 1, 0: -1}
    new_choice_vector = [choice_mapping[old_choice] for old_choice in choice]
    return new_choice_vector


def create_design_mat(choice, stim_left, stim_right, rewarded):
    # Create unnormalized_inpt: with first column = stim_right - stim_left,
    # second column as past choice, third column as WSLS
    stim = create_stim_vector(stim_left, stim_right)
    T = len(stim)
    design_mat = np.zeros((T, 3))
    design_mat[:, 0] = stim
    # make choice vector so that correct response for stim>0 is choice =1
    # and is 0 for stim <0 (viol is mapped to -1)
    choice = remap_choice_vals(choice)
    # create past choice vector:
    previous_choice, locs_mapping = create_previous_choice_vector(choice)
    # create wsls vector:
    wsls = create_wsls_covariate(previous_choice, rewarded, locs_mapping)
    # map previous choice to {-1,1}
    design_mat[:, 1] = 2 * previous_choice - 1
    design_mat[:, 2] = wsls
    return design_mat


def get_all_unnormalized_data_this_session(eid):
    # Load raw data
    animal, session_id, stim_left, stim_right, rewarded, choice, bias_probs \
        = get_raw_data(eid)
    # Subset choice and design_mat to 50-50 entries:
    trials_to_study = np.where(bias_probs == 0.5)[0]
    num_viols_50 = len(np.where(choice[trials_to_study] == 0)[0])
    if num_viols_50 < 10:
        # Create design mat = matrix of size T x 3, with entries for
        # stim/past choice/wsls
        unnormalized_inpt = create_design_mat(choice[trials_to_study],
                                              stim_left[trials_to_study],
                                              stim_right[trials_to_study],
                                              rewarded[trials_to_study])
        y = np.expand_dims(remap_choice_vals(choice[trials_to_study]), axis=1)
        session = [session_id for i in range(y.shape[0])]
        rewarded = np.expand_dims(rewarded[trials_to_study], axis=1)
    else:
        unnormalized_inpt = np.zeros((90, 3))
        y = np.zeros((90, 1))
        session = []
        rewarded = np.zeros((90, 1))
    return animal, unnormalized_inpt, y, session, num_viols_50, rewarded


def load_animal_list(file):
    container = np.load(file, allow_pickle=True)
    data = [container[key] for key in container]
    animal_list = data[0]
    return animal_list


def load_animal_eid_dict(file):
    with open(file, 'r') as f:
        animal_eid_dict = json.load(f)
    return animal_eid_dict


def load_data(animal_file):
    container = np.load(animal_file, allow_pickle=True)
    data = [container[key] for key in container]
    inpt = data[0]
    y = data[1]
    y = y.astype('int')
    session = data[2]
    return inpt, y, session


def create_train_test_sessions(session, num_folds=5):
    # create a session-fold lookup table
    num_sessions = len(np.unique(session))
    # Map sessions to folds:
    unshuffled_folds = np.repeat(np.arange(num_folds),
                                 np.ceil(num_sessions / num_folds))
    shuffled_folds = npr.permutation(unshuffled_folds)[:num_sessions]
    assert len(np.unique(
        shuffled_folds)) == 5, "require at least one session per fold for " \
                               "each animal!"
    # Look up table of shuffle-folds:
    sess_id = np.array(np.unique(session), dtype='str')
    shuffled_folds = np.array(shuffled_folds, dtype='O')
    session_fold_lookup_table = np.transpose(
        np.vstack([sess_id, shuffled_folds]))
    return session_fold_lookup_table


In [3]:
DOWNLOAD_DATA = True  # change to True to download raw data (WARNING: this
# can take a while)

ibl_data_path = "../../data/ibl/"
if DOWNLOAD_DATA: # Warning: this step takes a while
    if not os.path.exists(ibl_data_path):
        os.makedirs(ibl_data_path)
    # download IBL data
    url = 'https://ndownloader.figshare.com/files/21623715'
    wget.download(url, ibl_data_path)
    # now unzip downloaded data:
    with ZipFile(ibl_data_path + "ibl-behavior-data-Dec2019.zip",
                    'r') as zipObj:
        # extract all the contents of zip file in ibl_data_path
        zipObj.extractall(ibl_data_path)

# create directory for saving data:
if not os.path.exists(ibl_data_path + "partially_processed/"):
    os.makedirs(ibl_data_path + "partially_processed/")

# change directory so that ONE searches in correct directory:
os.chdir(ibl_data_path)
one = ONE()
eids = one.search(['_ibl_trials.*'])
assert len(eids) > 0, "ONE search is in incorrect directory"
animal_list = []
animal_eid_dict = defaultdict(list)

for eid in eids:
    bias_probs = one.load_dataset(eid, '_ibl_trials.probabilityLeft')
    comparison = np.unique(bias_probs) == np.array([0.2, 0.5, 0.8])
    # sessions with bias blocks
    if isinstance(comparison, np.ndarray):
        # update def of comparison to single True/False
        comparison = comparison.all()
    if comparison == True:
        animal = get_animal_name(eid)
        if animal not in animal_list:
            animal_list.append(animal)
        animal_eid_dict[animal].append(eid)

j = json.dumps(animal_eid_dict)
f = open("partially_processed/animal_eid_dict.json",  "w")
f.write(j)
f.close()

np.savez('partially_processed/animal_list.npz', animal_list)

c:\Users\Jasmine\miniconda3\envs\glmhmm\lib\site-packages\ipykernel_launcher.py:31: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.


In [16]:
data_dir = '../../data/ibl/'
# Create directories for saving data:
processed_ibl_data_path = data_dir + "data_for_cluster/"
if not os.path.exists(processed_ibl_data_path):
    os.makedirs(processed_ibl_data_path)
# Also create a subdirectory for storing each individual animal's data:
if not os.path.exists(processed_ibl_data_path + "data_by_animal/"):
    os.makedirs(processed_ibl_data_path + "data_by_animal/")

# Load animal list/results of partial processing:
animal_list = load_animal_list(
    data_dir + 'partially_processed/animal_list.npz')
animal_eid_dict = load_animal_eid_dict(
    data_dir + 'partially_processed/animal_eid_dict.json')

# Require that each animal has at least 30 sessions (=2700 trials) of data:
req_num_sessions = 30  # 30*90 = 2700
for animal in animal_list:
    num_sessions = len(animal_eid_dict[animal])
    if num_sessions < req_num_sessions:
        animal_list = np.delete(animal_list,
                                np.where(animal_list == animal))
# Identify idx in master array where each animal's data starts and ends:
animal_start_idx = {}
animal_end_idx = {}

final_animal_eid_dict = defaultdict(list)
# WORKHORSE: iterate through each animal and each animal's set of eids;
# obtain unnormalized data.  Write out each animal's data and then also
# write to master array
for z, animal in enumerate(animal_list):
    sess_counter = 0
    for eid in animal_eid_dict[animal]:
        animal, unnormalized_inpt, y, session, num_viols_50, rewarded = \
            get_all_unnormalized_data_this_session(
                eid)
        if num_viols_50 < 10:  # only include session if number of viols
            # in 50-50 block is less than 10
            if sess_counter == 0:
                animal_unnormalized_inpt = np.copy(unnormalized_inpt)
                animal_y = np.copy(y)
                animal_session = session
                animal_rewarded = np.copy(rewarded)
            else:
                animal_unnormalized_inpt = np.vstack(
                    (animal_unnormalized_inpt, unnormalized_inpt))
                animal_y = np.vstack((animal_y, y))
                animal_session = np.concatenate((animal_session, session))
                animal_rewarded = np.vstack((animal_rewarded, rewarded))
            sess_counter += 1
            final_animal_eid_dict[animal].append(eid)
    # Write out animal's unnormalized data matrix:
    np.savez(
        processed_ibl_data_path + 'data_by_animal/' + animal +
        '_unnormalized.npz',
        animal_unnormalized_inpt, animal_y,
        animal_session)
    animal_session_fold_lookup = create_train_test_sessions(animal_session,
                                                            5)
    np.savez(
        processed_ibl_data_path + 'data_by_animal/' + animal +
        "_session_fold_lookup" +
        ".npz",
        animal_session_fold_lookup)
    np.savez(
        processed_ibl_data_path + 'data_by_animal/' + animal +
        '_rewarded.npz',
        animal_rewarded)
    assert animal_rewarded.shape[0] == animal_y.shape[0]
    # Now create or append data to master array across all animals:
    if z == 0:
        master_inpt = np.copy(animal_unnormalized_inpt)
        animal_start_idx[animal] = 0
        animal_end_idx[animal] = master_inpt.shape[0] - 1
        master_y = np.copy(animal_y)
        master_session = animal_session
        master_session_fold_lookup_table = animal_session_fold_lookup
        master_rewarded = np.copy(animal_rewarded)
    else:
        animal_start_idx[animal] = master_inpt.shape[0]
        master_inpt = np.vstack((master_inpt, animal_unnormalized_inpt))
        animal_end_idx[animal] = master_inpt.shape[0] - 1
        master_y = np.vstack((master_y, animal_y))
        master_session = np.concatenate((master_session, animal_session))
        master_session_fold_lookup_table = np.vstack(
            (master_session_fold_lookup_table, animal_session_fold_lookup))
        master_rewarded = np.vstack((master_rewarded, animal_rewarded))
# Write out data from across animals
assert np.shape(master_inpt)[0] == np.shape(master_y)[
    0], "inpt and y not same length"
assert np.shape(master_rewarded)[0] == np.shape(master_y)[
    0], "rewarded and y not same length"
assert len(np.unique(master_session)) == \
        np.shape(master_session_fold_lookup_table)[
            0], "number of unique sessions and session fold lookup don't " \
                "match"
assert len(master_inpt) == 181530, "design matrix for all IBL animals " \
                                    "should have shape (181530, 3)"
assert len(animal_list) == 37, "37 animals were studied in Ashwood et " \
                                "al. (2020)"
normalized_inpt = np.copy(master_inpt)
normalized_inpt[:, 0] = preprocessing.scale(normalized_inpt[:, 0])
np.savez(processed_ibl_data_path + 'all_animals_concat' + '.npz',
            normalized_inpt,
            master_y, master_session)
np.savez(
    processed_ibl_data_path + 'all_animals_concat_unnormalized' + '.npz',
    master_inpt, master_y, master_session)
np.savez(
    processed_ibl_data_path + 'all_animals_concat_session_fold_lookup' +
    '.npz',
    master_session_fold_lookup_table)
np.savez(processed_ibl_data_path + 'all_animals_concat_rewarded' + '.npz',
            master_rewarded)
np.savez(processed_ibl_data_path + 'data_by_animal/' + 'animal_list.npz',
            animal_list)

j = json.dumps(final_animal_eid_dict)
f = open(processed_ibl_data_path + "final_animal_eid_dict.json", "w")
f.write(j)
f.close()

# Now write out normalized data (when normalized across all animals) for
# each animal:
counter = 0
for animal in animal_start_idx.keys():
    start_idx = animal_start_idx[animal]
    end_idx = animal_end_idx[animal]
    inpt = normalized_inpt[range(start_idx, end_idx + 1)]
    y = master_y[range(start_idx, end_idx + 1)]
    session = master_session[range(start_idx, end_idx + 1)]
    counter += inpt.shape[0]
    np.savez(processed_ibl_data_path + 'data_by_animal/' + animal + '_processed.npz',
                inpt, y,
                session)

assert counter == master_inpt.shape[0]

angelakilab/Subjects/IBL-T1/2019-03-18/001
angelakilab/Subjects/IBL-T1/2019-03-19/001
angelakilab/Subjects/IBL-T1/2019-03-20/002
angelakilab/Subjects/IBL-T1/2019-03-21/001
angelakilab/Subjects/IBL-T1/2019-03-22/001
angelakilab/Subjects/IBL-T1/2019-03-25/002
angelakilab/Subjects/IBL-T1/2019-03-29/001
angelakilab/Subjects/IBL-T1/2019-04-02/001
angelakilab/Subjects/IBL-T1/2019-04-03/001
angelakilab/Subjects/IBL-T1/2019-04-04/001
angelakilab/Subjects/IBL-T1/2019-04-05/001
angelakilab/Subjects/IBL-T1/2019-04-08/001
angelakilab/Subjects/IBL-T1/2019-04-09/001
angelakilab/Subjects/IBL-T1/2019-04-10/001
angelakilab/Subjects/IBL-T1/2019-04-11/001
angelakilab/Subjects/IBL-T1/2019-04-12/002
angelakilab/Subjects/IBL-T1/2019-04-15/001
angelakilab/Subjects/IBL-T1/2019-04-17/001
angelakilab/Subjects/IBL-T1/2019-04-18/001
angelakilab/Subjects/IBL-T1/2019-04-19/002
angelakilab/Subjects/IBL-T1/2019-04-22/001
angelakilab/Subjects/IBL-T1/2019-04-24/001
angelakilab/Subjects/IBL-T1/2019-04-25/001
angelakilab

In [4]:
ibl_data_path = "../../data/ibl/"
animal_eid_dict = load_animal_eid_dict(
    ibl_data_path + 'data_for_cluster/final_animal_eid_dict.json')
# must change directory for working with ONE
os.chdir(ibl_data_path)
one = ONE()

data_dir = 'response_times/data_by_animal/'
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

for animal in animal_eid_dict.keys():
    print(animal)
    animal_inpt, animal_y, animal_session = load_data(
        'data_for_cluster/data_by_animal/' + animal + '_processed.npz')
    for z, eid in enumerate(animal_eid_dict[animal]):
        raw_session_id = eid.split('Subjects/')[1]
        session_id = raw_session_id.replace('/', '-')
        full_sess_len = len(one.load_dataset(eid, '_ibl_trials.choice'))

        file_names = [
            '_ibl_trials.feedback_times', '_ibl_trials.response_times',
            '_ibl_trials.goCue_times', '_ibl_trials.stimOn_times'
        ]

        save_vars = [
            'feedback_times', 'response_times', 'go_cues', 'stim_on_times'
        ]

        for i, file in enumerate(file_names):
            full_path = 'ibl-behavioral-data-Dec2019/' + eid + \
                                '/alf/' + file + '.npy'
            if os.path.exists(full_path):
                globals()[save_vars[i]] = one.load_dataset(eid, file)
            else:
                globals()[save_vars[i]] = np.empty((full_sess_len, ))
                globals()[save_vars[i]][:] = np.nan

        start = np.nanmin(np.c_[stim_on_times, go_cues], axis=1)

        if (len(feedback_times) == len(response_times)): # some response
            # times/feedback times are missing, so fill these as best as
            # possible
            end = np.nanmin(np.c_[feedback_times, response_times], axis=1)
        elif len(feedback_times) == full_sess_len:
            end = feedback_times
        elif len(response_times) == full_sess_len:
            end = response_times

        # check timestamps increasing:
        idx_to_change = np.where(start > end)[0]

        if len(idx_to_change) > 0:
            start[idx_to_change[0]] = np.nan
            end[idx_to_change[0]] = np.nan

        # Check we have times for at least some trials
        nan_trial = np.isnan(np.c_[start, end]).any(axis=1)

        is_increasing = (((start < end) | nan_trial).all() and
                ((np.diff(start) > 0) | np.isnan(
                    np.diff(start))).all())

        if is_increasing and ~nan_trial.all() and len(start) == \
                full_sess_len and len(end) == full_sess_len: #
            # check that times are increasing and that len(start) ==
            # full_sess_len etc
            prob_left_dta = one.load_dataset(
                eid, '_ibl_trials.probabilityLeft')
            assert start.shape[0] == prob_left_dta.shape[0],\
                "different lengths for prob left and raw response dta: " + \
                str(start.shape[0]) + " vs " + str(
                    prob_left_dta.shape[0])

            # subset to trials corresponding to prob_left == 0.5:
            unbiased_idx = np.where(prob_left_dta == 0.5)
            response_dta = end[unbiased_idx] - start[unbiased_idx]

            if ((np.nanmedian(response_dta) >= 10) | (np.nanmedian(
                    response_dta) == np.nan)): # check that median
                # response time for session is less than 10 seconds
                response_dta = np.array([np.nan for i in range(len(
                    unbiased_idx[0]))])

            rt_sess = [session_id for i in range(response_dta.shape[0])]
            # before saving, confirm that there are as many trials as in
            # some of the other data:
            assert len(rt_sess) == animal_inpt[np.where(animal_session ==
                                                        session_id),
                                    :].shape[1], "response dta is different " \
                                                "shape compared to inpt"
        else: # if any of the conditions above fail, fill the session's
            # data with nans
            len_prob_50 = animal_inpt[np.where(animal_session ==
                                                        session_id),
                            :].shape[1]
            response_dta = np.array([np.nan for i in range(len_prob_50)])
            rt_sess = [session_id for i in range(response_dta.shape[0])]

        if z == 0:
            rt_session_dta_this_animal = rt_sess
            response_dta_this_animal = response_dta
        else:
            rt_session_dta_this_animal = np.concatenate(
                (rt_session_dta_this_animal, rt_sess))
            response_dta_this_animal = np.concatenate(
                (response_dta_this_animal, response_dta))

    assert len(response_dta_this_animal) == len(animal_inpt), "different size for response times and inpt"
    np.savez(data_dir + animal + '.npz', response_dta_this_animal,
                rt_session_dta_this_animal)

IBL-T1


c:\Users\Jasmine\miniconda3\envs\glmhmm\lib\site-packages\ipykernel_launcher.py:60: RuntimeWarning: invalid value encountered in less
c:\Users\Jasmine\miniconda3\envs\glmhmm\lib\site-packages\ipykernel_launcher.py:61: RuntimeWarning: invalid value encountered in greater


IBL-T2
IBL-T3
IBL-T4


c:\Users\Jasmine\miniconda3\envs\glmhmm\lib\site-packages\ipykernel_launcher.py:39: RuntimeWarning: All-NaN slice encountered
c:\Users\Jasmine\miniconda3\envs\glmhmm\lib\site-packages\ipykernel_launcher.py:51: RuntimeWarning: invalid value encountered in greater


NYU-01
NYU-02
NYU-04
NYU-06


c:\Users\Jasmine\miniconda3\envs\glmhmm\lib\site-packages\ipykernel_launcher.py:44: RuntimeWarning: All-NaN slice encountered


CSHL_001
CSHL_002
CSHL_003
CSHL_005
CSHL_007
CSHL_008
CSHL_010
CSHL_012
CSHL_014
CSHL_015
KS003
KS005
KS019
ZM_1367
ZM_1369
ZM_1371
ZM_1372
ZM_1743
ZM_1745


c:\Users\Jasmine\miniconda3\envs\glmhmm\lib\site-packages\numpy\lib\nanfunctions.py:1116: RuntimeWarning: All-NaN slice encountered
  overwrite_input=overwrite_input)


ZM_1746
ibl_witten_04
ibl_witten_05
ibl_witten_06
ibl_witten_07
ibl_witten_12
ibl_witten_13
ibl_witten_14
ibl_witten_15
ibl_witten_16
